# GemmaFit — Cloud Training Pipeline

**Target**: Fine-tune Gemma 4 (E2B / E4B) for trustworthy multi-exercise coaching.
**Platform**: Colab Pro (A100 40GB).
**Disk strategy**: HuggingFace `streaming=True` + on-the-fly extraction. Zero large files persisted.

## Phases

| § | Phase | Runs today? |
| --- | --- | --- |
| 1 | Environment + secrets | ✅ |
| 2 | Phase A — FC training data (streaming) | ✅ today |
| 3 | Phase B — Video → landmark extraction | ✅ today |
| 4 | Phase C — Gemini synthetic expansion | ⏸ tomorrow |
| 5 | Phase D — Load Gemma + Unsloth QLoRA | ✅ today (dry run) |
| 6 | Phase E — Training loop | ⏸ once data is final |
| 7 | Phase F — Save + export adapter | ⏸ |
| 8 | Phase G — Convert to GGUF for deployment | ⏸ |

## Output

Only the LoRA adapter (~50 MB) and final GGUF (~2 GB) ever leave the cloud.
Source datasets stream from HuggingFace Hub and are never persisted to disk.

## Reading order

Read the markdown cells once top-to-bottom, then execute §1 → §1.5 → §2 → §3 → §5 today. Skip §4, §6-8 — they wait for tomorrow's data + decisions.

## §1. Environment + Secrets

In [1]:
%%capture
# Install — version pinning matters for Gemma 4 + Unsloth (Daniel Han / Unsloth tested combo).
try:
    import numpy, PIL
    _numpy = f"numpy=={numpy.__version__}"
    _pil   = f"pillow=={PIL.__version__}"
except Exception:
    _numpy, _pil = "numpy", "pillow"

!uv pip install -qqq \
    "torch>=2.8.0" "triton>=3.4.0" {_numpy} {_pil} torchvision bitsandbytes \
    unsloth "unsloth_zoo>=2026.4.6" "transformers==5.5.0" torchcodec timm \
    datasets huggingface_hub hf_transfer mediapipe opencv-python yt-dlp trl peft accelerate

import torch
print('CUDA:', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
# Expected: A100 40GB (Colab Pro) or 2× Tesla T4 (Kaggle Notebooks)


In [2]:
# Colab secrets — set these in the side panel (🔑 icon) before running:
#   HF_TOKEN       — your HuggingFace token (read access; required for gated Gemma models)
#   GEMINI_API_KEY — for §4 only (defer until tomorrow)

from google.colab import userdata, drive
import os

os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN') or ''
# os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')   # tomorrow

# Mount Drive first because this notebook is configured to persist BOTH:
#   1. Hugging Face downloaded base-model snapshots/cache
#   2. trained outputs: checkpoints, LoRA adapters, merged exports, logs
# Warning: base models can consume many GB of Drive storage.
drive.mount('/content/drive')

WORKDIR = '/content/drive/MyDrive/GemmaFit_train'
HF_CACHE_DIR = f'{WORKDIR}/hf_cache'
BASE_MODEL_DIR = f'{WORKDIR}/models/base_models'
TRAINED_OUTPUT_DIR = f'{WORKDIR}/trained_outputs'

for d in [
    WORKDIR,
    HF_CACHE_DIR,
    BASE_MODEL_DIR,
    TRAINED_OUTPUT_DIR,
    f'{WORKDIR}/landmarks',
    f'{TRAINED_OUTPUT_DIR}/checkpoints',
    f'{TRAINED_OUTPUT_DIR}/adapter',
    f'{TRAINED_OUTPUT_DIR}/merged_fp16',
    f'{TRAINED_OUTPUT_DIR}/logs',
]:
    os.makedirs(d, exist_ok=True)

# Faster / cleaner Hugging Face downloads.
# HF_HOME controls the Hugging Face cache location. Here it is intentionally placed on Drive.
os.environ['HF_HOME'] = HF_CACHE_DIR
os.environ.setdefault('HF_HUB_ENABLE_HF_TRANSFER', '1')
os.environ.setdefault('HF_HUB_DISABLE_TELEMETRY', '1')

print('Workdir ready        :', WORKDIR)
print('HF cache on Drive    :', os.environ['HF_HOME'])
print('Base model directory :', BASE_MODEL_DIR)
print('Trained output dir   :', TRAINED_OUTPUT_DIR)


Mounted at /content/drive
Workdir ready        : /content/drive/MyDrive/GemmaFit_train
HF cache on Drive    : /content/drive/MyDrive/GemmaFit_train/hf_cache
Base model directory : /content/drive/MyDrive/GemmaFit_train/models/base_models
Trained output dir   : /content/drive/MyDrive/GemmaFit_train/trained_outputs


In [5]:
# Pull your existing 600 FC examples from your GitHub repo so the cloud has them.
# (Replace REPO_RAW with your actual public-raw URL once pushed.)

REPO_RAW = 'https://raw.githubusercontent.com/bkttt0429/GemmaFit/main'
FC_LOCAL = '/content/fc_training_data.json'

!curl -sL {REPO_RAW}/finetune/data/fc_training_data.json -o {FC_LOCAL}
!ls -lh {FC_LOCAL}

import json
fc = json.load(open(FC_LOCAL))
print('Local seeds:', fc['meta'])

-rw-r--r-- 1 root root 452K May  2 19:36 /content/fc_training_data.json
Local seeds: {'total': 600, 'train': 510, 'validation': 90, 'val_ratio': 0.15, 'seed': 42, 'generated_at': '2026-05-01', 'rule_version': 'prototype_threshold'}


## §1.5. Smart Hugging Face model selection + Drive auto-download

Run this after §1 install/secrets and before §5.

This cell does four things:

1. Detects GPU count, VRAM, RAM and free disk.
2. Chooses the safest Gemma model for the current runtime.
3. Checks Hugging Face token access before model loading.
4. Pre-downloads the selected model with `snapshot_download` into Google Drive, so §5 loads from a persisted local snapshot path.

The downstream cells use these globals:

| Global | Purpose |
| --- | --- |
| `RECOMMENDED_MODEL` | selected Hugging Face repo id |
| `LOCAL_MODEL_PATH` | Drive snapshot path if pre-download succeeds; otherwise repo id |
| `RECOMMENDED_SEQ_LEN` | max sequence length based on VRAM |
| `RECOMMENDED_BATCH` | per-device batch size based on VRAM |
| `GRAD_ACCUM` | gradient accumulation |
| `DEVICE_MAP` | `balanced` for multi-GPU, otherwise `None` |
| `TRAINED_OUTPUT_DIR` | Drive directory for checkpoints, adapters, merged exports and logs |
| `EST_STEPS_PER_SEC` | rough runtime estimate |

Default behavior in this notebook:

```python
SMART_DOWNLOAD_TO_DRIVE = True
```

So both the base model download and trained artifacts stay under:

```text
/content/drive/MyDrive/GemmaFit_train/
```


In [6]:
# Smart Hugging Face model selection + Drive auto-download
import os, shutil, json, time, torch
from huggingface_hub import snapshot_download, HfApi
from huggingface_hub.utils import HfHubHTTPError, GatedRepoError, RepositoryNotFoundError

print('=== Smart HF model setup ===')

# ── User knobs ─────────────────────────────────────────────────────────────
PREFER_SMALL_MODEL = False          # True = faster dev: prefer E2B even on bigger GPUs
SMART_DOWNLOAD = True               # True = pre-download selected model before §5
SMART_DOWNLOAD_TO_DRIVE = True      # User setting: persist base model snapshot in Google Drive
HF_MODEL_ALLOW_PATTERNS = [
    '*.json', '*.model', '*.txt', '*.safetensors',
    'tokenizer*', 'generation_config.json', 'chat_template*',
]

# Optional manual override. Leave None for auto mode.
MANUAL_MODEL_NAME = None
# MANUAL_MODEL_NAME = 'unsloth/gemma-4-E2B-it'
# MANUAL_MODEL_NAME = 'unsloth/gemma-4-E4B-it'
# MANUAL_MODEL_NAME = 'google/gemma-4-E4B-it'  # canonical gated fp16; slower/heavier

HF_TOKEN = os.environ.get('HF_TOKEN') or None
WORKDIR = globals().get('WORKDIR', '/content/drive/MyDrive/GemmaFit_train')
BASE_MODEL_DIR = globals().get('BASE_MODEL_DIR', f'{WORKDIR}/models/base_models')
TRAINED_OUTPUT_DIR = globals().get('TRAINED_OUTPUT_DIR', f'{WORKDIR}/trained_outputs')
HF_CACHE_DIR = globals().get('HF_CACHE_DIR', f'{WORKDIR}/hf_cache')

for d in [WORKDIR, BASE_MODEL_DIR, TRAINED_OUTPUT_DIR, HF_CACHE_DIR]:
    os.makedirs(d, exist_ok=True)
os.environ['HF_HOME'] = HF_CACHE_DIR

# ── Runtime detection ──────────────────────────────────────────────────────
def gib(x):
    return x / (1024 ** 3)

n_gpu = torch.cuda.device_count() if torch.cuda.is_available() else 0
gpu_names = [torch.cuda.get_device_name(i) for i in range(n_gpu)]
gpu_vram_gb = []
for i in range(n_gpu):
    props = torch.cuda.get_device_properties(i)
    gpu_vram_gb.append(gib(props.total_memory))

free_content_gb = gib(shutil.disk_usage('/content').free)
free_drive_gb = gib(shutil.disk_usage('/content/drive').free) if os.path.exists('/content/drive') else 0

print(f'GPU count       : {n_gpu}')
print(f'GPU names       : {gpu_names or ["CPU only"]}')
print(f'VRAM per GPU    : {[round(v, 1) for v in gpu_vram_gb]} GB')
print(f'Free /content   : {free_content_gb:.1f} GB')
print(f'Free Drive      : {free_drive_gb:.1f} GB')

# ── Model choice policy ────────────────────────────────────────────────────
max_vram = max(gpu_vram_gb) if gpu_vram_gb else 0
if MANUAL_MODEL_NAME:
    selected = MANUAL_MODEL_NAME
elif PREFER_SMALL_MODEL or max_vram < 18:
    selected = 'unsloth/gemma-4-E2B-it'
elif max_vram >= 24:
    selected = 'unsloth/gemma-4-E4B-it'
else:
    selected = 'unsloth/gemma-4-E2B-it'

# Training config policy.
if max_vram >= 40:
    seq_len, batch, grad_accum, est_sps = 2048, 4, 4, 6.0
elif max_vram >= 24:
    seq_len, batch, grad_accum, est_sps = 2048, 2, 8, 3.0
elif max_vram >= 15:
    seq_len, batch, grad_accum, est_sps = 1536, 1, 16, 1.5
else:
    seq_len, batch, grad_accum, est_sps = 1024, 1, 16, 0.7

device_map = 'balanced' if n_gpu > 1 else None

print('\nSelected model  :', selected)
print('Seq len / batch :', seq_len, '/', batch)
print('Grad accum      :', grad_accum)
print('Device map      :', device_map)

# ── Token / repo access check ──────────────────────────────────────────────
api = HfApi(token=HF_TOKEN)
try:
    info = api.model_info(selected)
    gated = getattr(info, 'gated', False)
    print(f'HF repo check   : OK  (gated={gated})')
except (GatedRepoError, RepositoryNotFoundError, HfHubHTTPError) as e:
    print('HF repo check   : FAILED')
    print('Reason          :', type(e).__name__, str(e)[:300])
    raise RuntimeError(
        'Cannot access selected model. Check HF_TOKEN, accept the model license on Hugging Face if gated, '
        'or set MANUAL_MODEL_NAME to an accessible repo.'
    )

# ── Smart pre-download ─────────────────────────────────────────────────────
local_model_path = selected
if SMART_DOWNLOAD:
    root = BASE_MODEL_DIR if SMART_DOWNLOAD_TO_DRIVE else '/content/hf_models'
    local_dir = os.path.join(root, selected.replace('/', '__'))
    os.makedirs(local_dir, exist_ok=True)

    if SMART_DOWNLOAD_TO_DRIVE and free_drive_gb < 20:
        print('\n⚠ Drive free space is below 20 GB. Base-model download may fail or fill Drive.')

    print('\nDownloading / syncing model snapshot...')
    print('Target          :', local_dir)
    print('Persist mode    :', 'Google Drive' if SMART_DOWNLOAD_TO_DRIVE else '/content temporary cache')
    print('HF cache        :', os.environ.get('HF_HOME'))
    t0 = time.time()
    try:
        local_model_path = snapshot_download(
            repo_id=selected,
            token=HF_TOKEN,
            local_dir=local_dir,
            local_dir_use_symlinks=False,
            allow_patterns=HF_MODEL_ALLOW_PATTERNS,
            resume_download=True,
            max_workers=8,
        )
        elapsed = time.time() - t0
        print(f'Download ready  : {local_model_path}')
        print(f'Elapsed         : {elapsed/60:.1f} min')
    except Exception as e:
        print('Download failed :', type(e).__name__, str(e)[:500])
        print('Fallback        : §5 will load directly from Hugging Face repo id.')
        local_model_path = selected

# ── Export globals used downstream ─────────────────────────────────────────
RECOMMENDED_MODEL = selected
LOCAL_MODEL_PATH = local_model_path
RECOMMENDED_SEQ_LEN = seq_len
RECOMMENDED_BATCH = batch
GRAD_ACCUM = grad_accum
DEVICE_MAP = device_map
EST_STEPS_PER_SEC = est_sps

SMART_HF_CONFIG = {
    'recommended_model': RECOMMENDED_MODEL,
    'local_model_path': LOCAL_MODEL_PATH,
    'seq_len': RECOMMENDED_SEQ_LEN,
    'batch': RECOMMENDED_BATCH,
    'grad_accum': GRAD_ACCUM,
    'device_map': DEVICE_MAP,
    'gpu_names': gpu_names,
    'vram_gb': gpu_vram_gb,
    'download_to_drive': SMART_DOWNLOAD_TO_DRIVE,
    'base_model_dir': BASE_MODEL_DIR,
    'hf_cache_dir': HF_CACHE_DIR,
    'trained_output_dir': TRAINED_OUTPUT_DIR,
}
print('\n=== Exported config ===')
print(json.dumps(SMART_HF_CONFIG, indent=2, ensure_ascii=False))


=== Smart HF model setup ===
GPU count       : 1
GPU names       : ['NVIDIA A100-SXM4-80GB']
VRAM per GPU    : [79.3] GB
Free /content   : 192.0 GB
Free Drive      : 182.4 GB

Selected model  : unsloth/gemma-4-E4B-it
Seq len / batch : 2048 / 4
Grad accum      : 4
Device map      : None
HF repo check   : OK  (gated=False)

Target          : /content/drive/MyDrive/GemmaFit_train/models/base_models/unsloth__gemma-4-E4B-it
Persist mode    : Google Drive
HF cache        : /content/drive/MyDrive/GemmaFit_train/hf_cache


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:189: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `snapshot_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

Download ready  : /content/drive/MyDrive/GemmaFit_train/models/base_models/unsloth__gemma-4-E4B-it
Elapsed         : 0.8 min

=== Exported config ===
{
  "recommended_model": "unsloth/gemma-4-E4B-it",
  "local_model_path": "/content/drive/MyDrive/GemmaFit_train/models/base_models/unsloth__gemma-4-E4B-it",
  "seq_len": 2048,
  "batch": 4,
  "grad_accum": 4,
  "device_map": null,
  "gpu_names": [
    "NVIDIA A100-SXM4-80GB"
  ],
  "vram_gb": [
    79.250732421875
  ],
  "download_to_drive": true,
  "base_model_dir": "/content/drive/MyDrive/GemmaFit_train/models/base_models",
  "hf_cache_dir": "/content/drive/MyDrive/GemmaFit_train/hf_cache",
  "trained_output_dir": "/content/drive/MyDrive/GemmaFit_train/trained_outputs"
}


## §2. Phase A — FC training data via HuggingFace streaming

Three sources, all streamed (zero disk):

| Source | Role | Mix weight |
| --- | --- | --- |
| Your 600 FC seeds | domain-specific GemmaFit behaviour | 30% |
| `glaiveai/glaive-function-calling-v2` | general FC schema robustness | 60% |
| `Anthropic/hh-rlhf` filtered | refusal style for out-of-scope queries | 10% |

Streaming means: nothing downloads. Each batch is fetched on demand during training. Disk usage stays at ~5 MB regardless of dataset size.

In [11]:
from datasets import load_dataset, interleave_datasets, IterableDataset

# 2A — Your domain seeds (small, can load eagerly)
domain_ds = load_dataset('json', data_files=FC_LOCAL, field='train', split='train')
print('Domain (eager):', len(domain_ds), 'examples')

# 2B — Glaive FC v2 (40K, stream)
glaive = load_dataset('glaiveai/glaive-function-calling-v2',
                      streaming=True, split='train')
print('Glaive (streaming): connected')

# 2C — HH-RLHF subset for refusal style (stream)
hh = load_dataset('Anthropic/hh-rlhf', streaming=True, split='train')
print('HH-RLHF (streaming): connected')

Generating train split: 0 examples [00:00, ? examples/s]

Domain (eager): 510 examples


README.md:   0%|          | 0.00/106 [00:00<?, ?B/s]

Glaive (streaming): connected


README.md: 0.00B [00:00, ?B/s]

HH-RLHF (streaming): connected


In [12]:
# Inspect one sample from each source
import itertools

print('=== Domain seed ===')
print(json.dumps(domain_ds[0], ensure_ascii=False, indent=2)[:600])
print()
print('=== Glaive ===')
print(json.dumps(next(iter(glaive)), ensure_ascii=False, indent=2)[:600])
print()
print('=== HH-RLHF ===')
print(json.dumps(next(iter(hh)), ensure_ascii=False, indent=2)[:600])

=== Domain seed ===
{
  "input": {
    "pattern": "bilateral_knee_dominant_sagittal",
    "phase": "bottom",
    "estimated_primary": [
      "quadriceps",
      "gluteus_maximus"
    ],
    "estimated_secondary": [
      "hamstrings",
      "calves",
      "core_stabilizers"
    ],
    "anomalies": [
      {
        "rule": 1,
        "joint": "left_knee",
        "ratio": 0.62,
        "severity": "severe",
        "evidence": "prototype_threshold",
        "left_angle": null,
        "right_angle": null,
        "difference": null,
        "deviation": null,
        "region": null,
        "note": null,
      

=== Glaive ===
{
  "system": "SYSTEM: You are a helpful assistant with access to the following functions. Use them if required -\n{\n    \"name\": \"get_exchange_rate\",\n    \"description\": \"Get the exchange rate between two currencies\",\n    \"parameters\": {\n        \"type\": \"object\",\n        \"properties\": {\n            \"base_currency\": {\n                \"ty

In [13]:
# Format each source into a unified chat schema:
#   {"messages": [{"role": "user", "content": ...},
#                  {"role": "assistant", "content": ...}]}

def fmt_domain(ex):
    """Your fc_training_data.json shape."""
    user = json.dumps(ex['input'], ensure_ascii=False)
    asst = json.dumps(ex['output'], ensure_ascii=False)
    return {'messages': [
        {'role': 'user',      'content': f'Motion report:\n{user}'},
        {'role': 'assistant', 'content': asst},
    ]}


def fmt_glaive(ex):
    """Glaive uses 'system' + 'chat' fields. Take the user/assistant turn."""
    sys_p = ex.get('system', '')
    chat = ex.get('chat', '')
    # crude split — better cleaning happens later
    return {'messages': [
        {'role': 'system',    'content': sys_p[:500]},
        {'role': 'user',      'content': chat[:500]},
        {'role': 'assistant', 'content': chat[500:1500]},
    ]}


def fmt_hh(ex):
    """HH-RLHF has 'chosen' (preferred) and 'rejected' (avoid) versions."""
    chosen = ex.get('chosen', '')
    parts = chosen.split('\n\nAssistant: ', 1)
    user_part = parts[0].replace('Human: ', '').strip()[:500]
    asst_part = (parts[1] if len(parts) > 1 else '').strip()[:500]
    return {'messages': [
        {'role': 'user',      'content': user_part},
        {'role': 'assistant', 'content': asst_part},
    ]}


domain_fmt = domain_ds.map(fmt_domain, remove_columns=domain_ds.column_names)
glaive_fmt = glaive.map(fmt_glaive, remove_columns=['system', 'chat'])
hh_fmt     = hh.map(fmt_hh, remove_columns=['chosen', 'rejected'])

# Convert eager domain → streaming so interleave_datasets accepts it
domain_iter = domain_fmt.to_iterable_dataset().shuffle(seed=42, buffer_size=1000)

mixed = interleave_datasets(
    [domain_iter, glaive_fmt, hh_fmt],
    probabilities=[0.30, 0.60, 0.10],
    stopping_strategy='all_exhausted',
)
print('Mixed stream ready (3 sources, weighted 30/60/10).')

Map:   0%|          | 0/510 [00:00<?, ? examples/s]

Mixed stream ready (3 sources, weighted 30/60/10).


In [14]:
# Sanity-check: take 5 samples from the mixed stream
for i, ex in enumerate(mixed.take(5)):
    print(f'[{i}]', json.dumps(ex, ensure_ascii=False)[:200])

[0] {"messages": [{"role": "system", "content": "SYSTEM: You are a helpful assistant with access to the following functions. Use them if required -\n{\n    \"name\": \"get_exchange_rate\",\n    \"descript
[1] {"messages": [{"role": "user", "content": "Motion report:\n{\"pattern\": \"prone_hands_shoulder_dominant_sagittal\", \"phase\": \"descent\", \"estimated_primary\": [\"pectoralis_major\", \"triceps\", 
[2] {"messages": [{"role": "system", "content": "SYSTEM: You are a helpful assistant with access to the following functions. Use them if required -\n{\n    \"name\": \"get_news_headlines\",\n    \"descrip
[3] {"messages": [{"role": "system", "content": "SYSTEM: You are a helpful assistant with access to the following functions. Use them if required -\n{\n    \"name\": \"generate_password\",\n    \"descript
[4] {"messages": [{"role": "system", "content": "SYSTEM: You are a helpful assistant with access to the following functions. Use them if required -\n{\n    \"name\": \"generate_pas

In [15]:
# Build an eval split from your existing 90 validation examples.
# Used for periodic eval during the overnight run so we get a learning curve,
# not just a training-loss plot.

eval_ds = load_dataset(
    'json', data_files=FC_LOCAL,
    field='validation' if 'validation' in fc else None,
    split='train',
)
print(f'Eval dataset: {len(eval_ds)} examples')

eval_fmt = eval_ds.map(fmt_domain, remove_columns=eval_ds.column_names)
print(f'Eval (formatted): {len(eval_fmt)} examples')


Generating train split: 0 examples [00:00, ? examples/s]

Eval dataset: 90 examples


Map:   0%|          | 0/90 [00:00<?, ? examples/s]

Eval (formatted): 90 examples


## §3. Phase B — Video → landmark extraction

Goal: get pose-landmark JSONL from public exercise videos **without storing the videos**.

Strategy:

```
URL list  →  yt-dlp tmp.mp4  →  MediaPipe pose  →  landmarks.jsonl  →  rm tmp.mp4
```

Each clip occupies disk for ~30 seconds, then is deleted. Final output is
small JSONL files (~10 KB per clip) on Drive. These feed the
**Phase 3 Safety & Trust benchmark** (verify gates fire correctly on real bodies).

In [16]:
# A starter list of public exercise videos. Add more as needed.
# Format: list of {url, exercise, view, source}

VIDEO_TARGETS = [
    # Squat — frontal
    {'url': 'https://commons.wikimedia.org/wiki/File:Squat-1.webm',
     'exercise': 'squat', 'view': 'frontal', 'source': 'wikimedia'},
    # Push-up
    {'url': 'https://commons.wikimedia.org/wiki/File:Push-ups.webm',
     'exercise': 'push_up', 'view': 'side', 'source': 'wikimedia'},
    # Add YouTube urls here, e.g.:
    # {'url': 'https://youtu.be/<id>', 'exercise': 'lunge', 'view': 'side', 'source': 'youtube'},
]
print(f'{len(VIDEO_TARGETS)} videos queued')

2 videos queued


In [17]:
import os, json, time, subprocess, hashlib
import cv2, mediapipe as mp

TMP_VIDEO = '/content/tmp_clip.mp4'
OUT_DIR = f'{WORKDIR}/landmarks'

LM_NAMES = [
    'nose','left_eye_inner','left_eye','left_eye_outer','right_eye_inner',
    'right_eye','right_eye_outer','left_ear','right_ear','mouth_left','mouth_right',
    'left_shoulder','right_shoulder','left_elbow','right_elbow',
    'left_wrist','right_wrist','left_pinky','right_pinky','left_index','right_index',
    'left_thumb','right_thumb','left_hip','right_hip','left_knee','right_knee',
    'left_ankle','right_ankle','left_heel','right_heel','left_foot_index','right_foot_index',
]

def download_clip(url: str, out_path: str = TMP_VIDEO) -> bool:
    if os.path.exists(out_path): os.remove(out_path)
    cmd = ['yt-dlp', '-q', '-f', 'best[height<=720]', '-o', out_path, url]
    r = subprocess.run(cmd, capture_output=True)
    return r.returncode == 0 and os.path.exists(out_path)


def extract_landmarks(video_path: str, sample_every: int = 2) -> list:
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    frames = []
    pose = mp.solutions.pose.Pose(min_detection_confidence=0.5,
                                  min_tracking_confidence=0.5)
    idx = 0
    while cap.isOpened():
        ok, img = cap.read()
        if not ok: break
        if idx % sample_every == 0:
            res = pose.process(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
            if res.pose_landmarks:
                frame = {'frame': idx, 'landmarks': {}}
                for i, lm in enumerate(res.pose_landmarks.landmark):
                    if i < len(LM_NAMES):
                        n = LM_NAMES[i]
                        frame['landmarks'][f'{n}_x'] = lm.x
                        frame['landmarks'][f'{n}_y'] = lm.y
                        frame['landmarks'][f'{n}_z'] = lm.z
                        frame['landmarks'][f'{n}_vis'] = lm.visibility
                frames.append(frame)
        idx += 1
    cap.release()
    pose.close()
    return frames, fps


def process_one(target: dict) -> dict:
    digest = hashlib.md5(target['url'].encode()).hexdigest()[:10]
    out_path = f'{OUT_DIR}/{target["exercise"]}_{digest}.jsonl'
    if os.path.exists(out_path):
        return {'status': 'cached', 'path': out_path}

    if not download_clip(target['url']):
        return {'status': 'download_failed'}
    try:
        frames, fps = extract_landmarks(TMP_VIDEO)
    except Exception as e:
        return {'status': f'extract_failed: {e}'}
    finally:
        if os.path.exists(TMP_VIDEO): os.remove(TMP_VIDEO)

    with open(out_path, 'w') as f:
        f.write(json.dumps({'meta': {**target, 'fps': fps,
                                      'frame_count': len(frames)}}) + '\n')
        for fr in frames:
            f.write(json.dumps(fr) + '\n')
    return {'status': 'ok', 'path': out_path, 'frames': len(frames)}


for tgt in VIDEO_TARGETS:
    t0 = time.time()
    res = process_one(tgt)
    dt = time.time() - t0
    print(f'[{dt:5.1f}s] {tgt["exercise"]:8s} {tgt["url"][:50]:50s} → {res}')

[  1.8s] squat    https://commons.wikimedia.org/wiki/File:Squat-1.we → {'status': 'download_failed'}
[  1.1s] push_up  https://commons.wikimedia.org/wiki/File:Push-ups.w → {'status': 'download_failed'}


In [18]:
# Inspect one extracted file
import os
files = [f for f in os.listdir(OUT_DIR) if f.endswith('.jsonl')]
print(f'{len(files)} landmark files:')
for f in files: print(f'  {f}: {os.path.getsize(os.path.join(OUT_DIR, f))/1024:.1f} KB')

if files:
    with open(os.path.join(OUT_DIR, files[0])) as fh:
        meta = json.loads(fh.readline())
        print()
        print('First file meta:', meta)

0 landmark files:


## §4. Phase C — Gemini synthetic expansion (TOMORROW)

Skip today. Plan:

1. Take each of your 600 FC seeds.
2. For each, ask Gemini Flash to rephrase the coaching message in 5 styles
   (formal / casual / encouraging / firm / multilingual zh-TW).
3. Write `/content/expanded_3000.jsonl` directly (no intermediate dump).
4. Cost estimate: 600 × 5 × ~500 tokens ≈ \$1-2 of Gemini API.

Why defer: the 600 + Glaive 40K is already enough to start a baseline run.
Spending API credits before you know whether the baseline works is premature.

In [19]:
# DRY-RUN STUB — uncomment when ready (see §4 reasoning above).
#
# import google.generativeai as genai
# genai.configure(api_key=os.environ['GEMINI_API_KEY'])
# model = genai.GenerativeModel('gemini-2.0-flash')
# ...
print('§4 deferred to tomorrow.')

§4 deferred to tomorrow.


## §5. Phase D — Load Gemma 4 + Unsloth QLoRA

Today: load model, dry-run a forward pass to confirm GPU memory budget.
Don't train yet — wait for §4 expanded data so we don't waste a training run.

Memory check on A100 40GB:
- Gemma 4 E2B in 4-bit ≈ 1.5 GB
- E4B in 4-bit ≈ 3 GB
- LoRA adapters: < 0.5 GB
- Activations + KV cache: 5-15 GB depending on seq_len
- Both fit comfortably on A100 40GB.

In [20]:
from unsloth import FastModel
import os, torch

# Use §1.5 smart Hugging Face setup if available; otherwise fall back.
MODEL_NAME = globals().get('RECOMMENDED_MODEL', 'unsloth/gemma-4-E4B-it')
MODEL_SOURCE = globals().get('LOCAL_MODEL_PATH', MODEL_NAME)   # local snapshot path when pre-downloaded
MAX_SEQ_LEN = globals().get('RECOMMENDED_SEQ_LEN', 2048)
DEVICE_MAP = globals().get('DEVICE_MAP', 'balanced' if torch.cuda.device_count() > 1 else None)

# Override here if you want a specific model regardless of detection:
# MODEL_NAME = 'unsloth/gemma-4-E4B-it'
# MODEL_SOURCE = MODEL_NAME
# MODEL_NAME = 'unsloth/gemma-4-E2B-it'
# MODEL_SOURCE = MODEL_NAME

print(f'Loading model source: {MODEL_SOURCE}')
print(f'Model label         : {MODEL_NAME}')
print(f'seq_len={MAX_SEQ_LEN}, device_map={DEVICE_MAP}')

model, tokenizer = FastModel.from_pretrained(
    model_name=MODEL_SOURCE,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
    full_finetuning=False,
    device_map=DEVICE_MAP,
    token=os.environ.get('HF_TOKEN') or None,
)
print(f'Loaded label        : {MODEL_NAME}')
print(f'Loaded source       : {MODEL_SOURCE}')
print(f'GPU mem after load  : {torch.cuda.memory_allocated()/1e9:.2f} GB')
print(f'GPU count           : {torch.cuda.device_count()} (device_map={DEVICE_MAP})')


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
Loading model source: /content/drive/MyDrive/GemmaFit_train/models/base_models/unsloth__gemma-4-E4B-it
Model label         : unsloth/gemma-4-E4B-it
seq_len=2048, device_map=None
==((====))==  Unsloth 2026.4.8: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.251 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/2130 [00:00<?, ?it/s]

Loaded label        : unsloth/gemma-4-E4B-it
Loaded source       : /content/drive/MyDrive/GemmaFit_train/models/base_models/unsloth__gemma-4-E4B-it
GPU mem after load  : 10.02 GB
GPU count           : 1 (device_map=None)


In [21]:
# Add LoRA adapters via FastModel
model = FastModel.get_peft_model(
    model,
    r=16,                # LoRA rank — 16 is a good default for E2B/E4B
    target_modules=['q_proj','k_proj','v_proj','o_proj',
                    'gate_proj','up_proj','down_proj'],
    lora_alpha=16,
    lora_dropout=0.0,    # 0 is fastest; raise to 0.05 if overfitting
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
    max_seq_length=MAX_SEQ_LEN,
)
trainable, total = 0, 0
for p in model.parameters():
    total += p.numel()
    if p.requires_grad: trainable += p.numel()
print(f'Trainable: {trainable/1e6:.2f}M / {total/1e9:.2f}B '
      f'({100*trainable/total:.2f}%)')


Trainable: 42.40M / 6.02B (0.70%)


In [23]:
# Quick generation sanity check on the BASE model (before any training).
FastModel.for_inference(model)

seed = domain_ds[0]
user_msg = f"Motion report:\n{json.dumps(seed['input'], ensure_ascii=False)}"
prompt = tokenizer.apply_chat_template(
    [{'role': 'user', 'content': user_msg}],
    tokenize=False, add_generation_prompt=True,
)
inputs = tokenizer(text=prompt, return_tensors='pt').to('cuda')
out = model.generate(**inputs, max_new_tokens=120, do_sample=False)
print('=== Base-model output (no fine-tune) ===')
print(tokenizer.decode(out[0][inputs['input_ids'].shape[1]:],
                       skip_special_tokens=True))
print()
print('=== Expected (your seed) ===')
print(json.dumps(seed['output'], ensure_ascii=False, indent=2))


=== Base-model output (no fine-tune) ===
This is a motion report generated from an analysis of a movement pattern, likely related to a squat or knee flexion exercise.

Here is a detailed breakdown and interpretation of the provided JSON data:

### Summary Interpretation

The analysis is focused on a **bilateral knee-dominant sagittal plane** movement, captured at the **"bottom"** phase (likely the deepest point of a squat). The primary muscles engaged are the **quadriceps** and **gluteus maximus**, with secondary involvement from the **hamstrings, calves, and core stabilizers**.

The most critical finding is a **severe

=== Expected (your seed) ===
{
  "function": "correct_knee_alignment",
  "args": {
    "side": "left",
    "ratio": 0.62,
    "severity": "severe",
    "joint": null,
    "left": null,
    "right": null,
    "deviation": null,
    "region": null,
    "pattern": null,
    "primary_muscles": null,
    "direction": null,
    "distance": null,
    "velocity": null,
    "cur

## §5.5. Pre-flight check (run before §6)

Catch bad configs BEFORE you go to sleep. This cell verifies:

1. Model loaded and on GPU.
2. Mixed dataset stream produces valid batches.
3. Tokenizer chat template renders correctly.
4. Drive is mounted and writable.
5. Eval set is loaded.

If any check fails, the cell raises — fix the issue before flipping
`RUN_TRAINING = True`.


In [24]:
import os, json, torch

print('=== Pre-flight check ===\n')

# 1. Model + GPU
assert 'model' in dir(), '❌ Model not loaded — re-run §5'
gpu_mem = torch.cuda.memory_allocated() / 1e9
assert gpu_mem > 0.5, f'❌ GPU memory unexpectedly low: {gpu_mem:.2f} GB'
print(f'✅ Model on GPU ({gpu_mem:.2f} GB allocated)')

# 2. Mixed stream produces samples
sample = next(iter(mixed.take(1)))
assert 'messages' in sample, '❌ Stream missing "messages" key'
assert len(sample['messages']) >= 2, '❌ Sample has <2 messages'
print(f'✅ Mixed stream produces valid samples ({len(sample["messages"])} msgs in first)')

# 3. Tokenizer chat template
formatted = tokenizer.apply_chat_template(
    sample['messages'], tokenize=False, add_generation_prompt=False)
assert len(formatted) > 50, f'❌ Empty chat template output: {formatted!r}'
print(f'✅ Chat template renders ({len(formatted)} chars)')

# 4. Drive is mounted + writable
test_file = f'{WORKDIR}/_preflight_test.txt'
with open(test_file, 'w') as f:
    f.write('ok')
os.remove(test_file)
print(f'✅ Drive writable: {WORKDIR}')

# 5. Eval set
assert 'eval_fmt' in dir(), '❌ eval_fmt not built — re-run the eval-split cell in §2'
assert len(eval_fmt) >= 10, f'❌ Eval set too small: {len(eval_fmt)}'
print(f'✅ Eval set ready ({len(eval_fmt)} examples)')

# 6. Estimate runtime
print('\n=== Runtime estimate ===')
n_gpu = torch.cuda.device_count()
gpu_name = torch.cuda.get_device_name(0)
if 'A100' in gpu_name:
    sps = 6.0   # rough: 6 steps/sec on A100 with 4B + bs=16 + seq=2048
elif 'T4' in gpu_name:
    sps = 1.5   # rough on Kaggle 2x T4
else:
    sps = 3.0
overnight_h = 5000 / sps / 3600
print(f'  GPU: {gpu_name} × {n_gpu}')
print(f'  Estimated: {sps:.1f} steps/sec → 5000 steps in ~{overnight_h:.1f} hr')

print('\n🟢 All checks passed. Set RUN_TRAINING=True in §6 to start overnight run.')


=== Pre-flight check ===

✅ Model on GPU (10.19 GB allocated)
✅ Mixed stream produces valid samples (3 msgs in first)
✅ Chat template renders (839 chars)
✅ Drive writable: /content/drive/MyDrive/GemmaFit_train
✅ Eval set ready (90 examples)

=== Runtime estimate ===
  GPU: NVIDIA A100-SXM4-80GB × 1
  Estimated: 6.0 steps/sec → 5000 steps in ~0.2 hr

🟢 All checks passed. Set RUN_TRAINING=True in §6 to start overnight run.


## §6. Phase E — Training loop

**Run this only after §4 (Gemini expansion) is done tomorrow.** The cell
below is the actual SFTTrainer setup; the safety check at the top stops
you from accidentally training on insufficient data.

In [25]:
# ── Overnight training with disconnect resilience ────────────────────────
# Tuned for an 8-12 hour Colab Pro session.

RUN_TRAINING = False   # ← flip to True before going to sleep

if not RUN_TRAINING:
    raise SystemExit('Training disabled. Set RUN_TRAINING=True before sleep.')

from trl import SFTTrainer, SFTConfig
from transformers import EarlyStoppingCallback
import os, glob, json, time, re

# Pick batch/accum from §1.5 (or sensible defaults)
BATCH       = globals().get('RECOMMENDED_BATCH', 4)
GRAD_ACCUM_ = globals().get('GRAD_ACCUM', 4)
SEQ_LEN     = globals().get('RECOMMENDED_SEQ_LEN', MAX_SEQ_LEN)

TRAINED_OUTPUT_DIR = globals().get('TRAINED_OUTPUT_DIR', f'{WORKDIR}/trained_outputs')
MODEL_TAG = re.sub(r'[^A-Za-z0-9_.-]+', '_', globals().get('MODEL_NAME', 'gemma_model'))
RUN_NAME = f'{MODEL_TAG}_gemmafit_v1'

CKPT_DIR = f'{TRAINED_OUTPUT_DIR}/checkpoints/{RUN_NAME}'
ADAPTER_PATH = f'{TRAINED_OUTPUT_DIR}/adapter/{RUN_NAME}_final_adapter'
LOG_DIR = f'{TRAINED_OUTPUT_DIR}/logs/{RUN_NAME}'

for d in [CKPT_DIR, ADAPTER_PATH, LOG_DIR]:
    os.makedirs(d, exist_ok=True)

existing = sorted(glob.glob(f'{CKPT_DIR}/checkpoint-*'),
                  key=lambda p: int(p.rsplit('-', 1)[1]))
RESUME = existing[-1] if existing else None
if RESUME:
    print(f'⚠ Resuming from {RESUME}')
else:
    print('Fresh overnight run.')

print(f'Batch: {BATCH} × accum {GRAD_ACCUM_} = effective {BATCH * GRAD_ACCUM_}')
print(f'Seq len: {SEQ_LEN}')
print(f'Checkpoint dir: {CKPT_DIR}')
print(f'Final adapter : {ADAPTER_PATH}')
print(f'Log dir       : {LOG_DIR}')

FastModel.for_training(model)

training_args = SFTConfig(
    output_dir=CKPT_DIR,
    per_device_train_batch_size=BATCH,
    gradient_accumulation_steps=GRAD_ACCUM_,
    learning_rate=2e-4,
    warmup_ratio=0.03,
    max_steps=5000,                          # ← overnight budget
    logging_steps=25,
    save_strategy='steps',
    save_steps=200,
    save_total_limit=5,
    eval_strategy='steps',
    eval_steps=500,
    per_device_eval_batch_size=BATCH,
    bf16=True,
    optim='adamw_8bit',
    seed=42,
    max_seq_length=SEQ_LEN,
    dataset_text_field=None,
    packing=False,
    report_to='none',
    logging_dir=LOG_DIR,
    resume_from_checkpoint=RESUME,
    ignore_data_skip=True,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
)

def fmt_for_sft(example):
    return tokenizer.apply_chat_template(
        example['messages'], tokenize=False, add_generation_prompt=False)

trainer = SFTTrainer(
    model=model,
    train_dataset=mixed,
    eval_dataset=eval_fmt,
    args=training_args,
    formatting_func=fmt_for_sft,
    tokenizer=tokenizer,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

t0 = time.time()
result = trainer.train(resume_from_checkpoint=RESUME)
elapsed = time.time() - t0

# Final-adapter save: trained output is persisted in Drive under TRAINED_OUTPUT_DIR.
model.save_pretrained(ADAPTER_PATH)
tokenizer.save_pretrained(ADAPTER_PATH)
trainer.save_state()

# DONE marker in the same trained output directory
done_marker = f'{TRAINED_OUTPUT_DIR}/TRAINING_DONE_{RUN_NAME}.json'
with open(done_marker, 'w') as f:
    json.dump({
        'finished_at': time.strftime('%Y-%m-%d %H:%M:%S'),
        'elapsed_seconds': int(elapsed),
        'final_train_loss': float(result.training_loss),
        'best_metric': result.metrics.get('best_metric', None),
        'global_step': result.global_step,
        'adapter_path': ADAPTER_PATH,
        'checkpoint_dir': CKPT_DIR,
        'log_dir': LOG_DIR,
        'effective_batch': BATCH * GRAD_ACCUM_,
        'seq_len': SEQ_LEN,
        'model': MODEL_NAME,
        'model_source': globals().get('MODEL_SOURCE', MODEL_NAME),
        'trained_output_dir': TRAINED_OUTPUT_DIR,
    }, f, indent=2)

print(f'\n{"="*50}')
print(f'  ✅ TRAINING DONE in {elapsed/3600:.2f} hr')
print(f'  Final train loss: {result.training_loss:.4f}')
print(f'  Steps completed:  {result.global_step}')
print(f'  Adapter saved:    {ADAPTER_PATH}')
print(f'  Marker file:      {done_marker}')
print(f'{"="*50}')


SystemExit: Training disabled. Set RUN_TRAINING=True before sleep.

/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


## §6.5. Reconnect-after-disconnect helper

Colab Pro sessions can drop after ~24 hr or on browser reload. Workflow:

1. **Reopen the notebook** → Runtime → Reconnect.
2. Re-run §1 (env + secrets + Drive mount).
3. Re-run §2 (data sources reconnect — streaming, no state to recover).
4. Re-run §5 (model loads fresh, ~2 min).
5. Re-run §5.5 (pre-flight check).
6. Re-run §6 with `RUN_TRAINING = True` → it auto-detects the latest
   checkpoint in Drive and resumes from there.

Skip §3 (landmarks) and §4 (synthetic) if those finished previously.

The cell below diagnoses your current state and detects whether
**training already finished** (via `TRAINING_DONE.json` marker).


In [26]:
import os, glob, json

CKPT_DIR = f'{WORKDIR}/checkpoints/gemma4_e4b_v1'
DONE_MARKER = f'{WORKDIR}/TRAINING_DONE.json'

# 1. Did training already finish?
if os.path.exists(DONE_MARKER):
    with open(DONE_MARKER) as f:
        info = json.load(f)
    print('🎉 Training already finished!')
    for k, v in info.items():
        print(f'  {k}: {v}')
    print('\n→ Skip to §7 (download) or §8 (GGUF convert).')
else:
    # 2. Otherwise show checkpoint state
    ckpts = sorted(glob.glob(f'{CKPT_DIR}/checkpoint-*'),
                   key=lambda p: int(p.rsplit('-', 1)[1]))
    print('=== Resume diagnostic ===')
    print(f'Drive workdir   : {WORKDIR}')
    print(f'Checkpoints dir : {CKPT_DIR}')
    print(f'Found {len(ckpts)} checkpoint(s):')
    for c in ckpts[-5:]:
        step = c.rsplit('-', 1)[1]
        size_mb = sum(os.path.getsize(os.path.join(c, f))
                      for f in os.listdir(c)
                      if os.path.isfile(os.path.join(c, f))) / 1e6
        print(f'  step {step:>6}  ({size_mb:.1f} MB)  {c}')
    if ckpts:
        print(f'\n➜ Re-run §1, §2, §5, §5.5, §6 — will resume from step '
              f'{ckpts[-1].rsplit(\"-\", 1)[1]}')
    else:
        print('  (none — fresh run)')


SyntaxError: unexpected character after line continuation character (4235145078.py, line 30)

## §7. Phase F — Save adapter, download to local

After training, only the LoRA adapter (~50 MB) gets persisted. Pull it back to your laptop manually from Drive.

In [27]:
# Save LoRA adapter to the same Drive trained-output directory
TRAINED_OUTPUT_DIR = globals().get('TRAINED_OUTPUT_DIR', f'{WORKDIR}/trained_outputs')
os.makedirs(f'{TRAINED_OUTPUT_DIR}/adapter', exist_ok=True)

ADAPTER_PATH = f'{TRAINED_OUTPUT_DIR}/adapter/gemma_fc_v1'
model.save_pretrained(ADAPTER_PATH)
tokenizer.save_pretrained(ADAPTER_PATH)
!du -sh {ADAPTER_PATH}
print('Adapter saved to:', ADAPTER_PATH)

# Optional: also push to HuggingFace Hub so you can pull from anywhere
# model.push_to_hub('<your-hf-username>/gemmafit-gemma4-fc-v1', token=os.environ['HF_TOKEN'])


Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/GemmaFit_train/trained_outputs/adapter/gemma_fc_v1/tokenizer_config.json.


193M	/content/drive/MyDrive/GemmaFit_train/trained_outputs/adapter/gemma_fc_v1
Adapter saved to: /content/drive/MyDrive/GemmaFit_train/trained_outputs/adapter/gemma_fc_v1


## §8. Phase G — Convert to GGUF for llama.cpp deployment

Final step: merge LoRA into base + quantize to GGUF Q5_K_M for desktop /
Q4_K_M for Android. Output is the file you'll ship to `models/`.

In [ ]:
# Merge LoRA into base + save fp16 to the same Drive trained-output directory
# Warning: merged fp16 can be large; make sure Drive has enough free space.
TRAINED_OUTPUT_DIR = globals().get('TRAINED_OUTPUT_DIR', f'{WORKDIR}/trained_outputs')
os.makedirs(f'{TRAINED_OUTPUT_DIR}/merged_fp16', exist_ok=True)

MERGED_PATH = f'{TRAINED_OUTPUT_DIR}/merged_fp16/gemmafit_merged_fp16_v1'
model.save_pretrained_merged(MERGED_PATH, tokenizer, save_method='merged_16bit')
!ls -lh {MERGED_PATH}
print('Merged model saved to:', MERGED_PATH)


Detected local model directory: /content/drive/MyDrive/GemmaFit_train/models/base_models/unsloth__gemma-4-E4B-it


Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/GemmaFit_train/trained_outputs/merged_fp16/gemmafit_merged_fp16_v1/tokenizer_config.json.


Found HuggingFace hub cache directory: /content/drive/MyDrive/GemmaFit_train/hf_cache/hub


Unsloth: Preparing safetensor model files:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
# Convert merged → GGUF via llama.cpp
!git clone --depth 1 https://github.com/ggerganov/llama.cpp /content/llama.cpp
!pip install -q -r /content/llama.cpp/requirements.txt

# fp16 GGUF
GGUF_FP16 = f'{WORKDIR}/gemmafit-fp16.gguf'
!python /content/llama.cpp/convert_hf_to_gguf.py {MERGED_PATH} \
    --outfile {GGUF_FP16} --outtype f16

# Quantize to Q5_K_M (desktop) and Q4_K_M (Android)
!cd /content/llama.cpp && cmake -B build && cmake --build build --target llama-quantize -j

GGUF_Q5 = f'{WORKDIR}/gemmafit-q5_k_m.gguf'
GGUF_Q4 = f'{WORKDIR}/gemmafit-q4_k_m.gguf'
!/content/llama.cpp/build/bin/llama-quantize {GGUF_FP16} {GGUF_Q5} Q5_K_M
!/content/llama.cpp/build/bin/llama-quantize {GGUF_FP16} {GGUF_Q4} Q4_K_M

# Drop the fp16 — it's huge and we don't need it after quant
!rm -f {GGUF_FP16}
!ls -lh {WORKDIR}/*.gguf

## §9. Reference — disk usage, troubleshooting, decisions

### Reference notebooks (Daniel Han / Unsloth)

| Notebook | Use |
| --- | --- |
| [Gemma 4 E4B — multimodal SFT](https://www.kaggle.com/code/danielhanchen/gemma4-e4b-unsloth) | Closest to our setup |
| [Gemma 4 31B — Kaggle 2× T4 fits](https://www.kaggle.com/code/danielhanchen/gemma4-31b-unsloth) | If you want the largest model |
| [Gemma 4 31B Vision SFT](https://www.kaggle.com/code/danielhanchen/gemma4-31b-vision-unsloth) | If pose-image multimodal needed |
| [All Unsloth notebooks](https://github.com/unslothai/notebooks/?tab=readme-ov-file#gemma-4-notebooks) | Audio / vision / text variants |

Optional tooling: [unsloth-buddy](https://github.com/TYH-labs/unsloth-buddy) — Claude Code skill that automates env setup, dataset reformatting, base-vs-FT comparison, and the side-by-side demo HTML the Kaggle judges expect.

### Disk usage budget

| Item | Size | Where |
| --- | --- | --- |
| Streaming buffers | < 100 MB | Colab `/tmp` (auto-cleared) |
| Domain seeds (your 600) | < 1 MB | `/content` |
| Landmark JSONL files | ~10 MB total | Drive `landmarks/` |
| Base 4-bit model | ~3 GB | GPU mem (not disk) |
| Checkpoints (3 kept) | ~150 MB | Drive `checkpoints/` |
| LoRA adapter (final) | ~50 MB | Drive `adapter/` |
| Merged fp16 (temp) | ~6 GB | Colab `/content` (delete after) |
| Final Q5_K_M GGUF | ~2 GB | Drive |
| Final Q4_K_M GGUF | ~1.5 GB | Drive |
| **Persistent total in Drive** | **~3.7 GB** | |

Your laptop only ever pulls back the GGUFs (~3.5 GB) and the LoRA (~50 MB).

### Troubleshooting

| Symptom | Likely cause | Fix |
| --- | --- | --- |
| `OOM at training start` | seq_len or batch too big | drop `max_seq_length` to 1024, batch to 2 |
| `Streaming dataset hangs` | HF Hub rate limit | retry, or set `HF_TOKEN` |
| `Loss = NaN` | learning rate too high | LR 1e-4 instead of 2e-4 |
| `Trained model still bad JSON` | not enough domain weight | raise domain mix from 30% → 50% |
| `GGUF quantize fails` | architecture not yet supported | use Q8_0 as fallback |
| `FastLanguageModel not found` | older Unsloth | switch to `FastModel` (Gemma 4 path) |
| `transformers version mismatch` | unpinned install | re-run §1 install cell with pinned versions |

### Decision points (for tomorrow)

Before flipping `RUN_TRAINING = True`:

1. Did Phase A's `mixed` stream produce 5 sane samples in §2's preview?
2. Did Phase B's landmark extraction produce ≥ 3 JSONL files?
3. Did §5 base-model generation roughly resemble the expected output?
   - If output is nonsense → fine-tuning will help most.
   - If output is already 70%+ correct → consider skipping fine-tune, jump to LiteRT spike.

### Hardware: A100 vs Kaggle 2× T4

| Setup | Pros | Cons |
| --- | --- | --- |
| Colab Pro A100 40GB | Single GPU simple, big batch, fast | $10/mo, 24h sessions |
| Kaggle 2× T4 (free) | Free, 30hr/week, `device_map="balanced"` works | Slower, must split layers |

This notebook auto-detects: if `torch.cuda.device_count() > 1`, sets `device_map="balanced"` for Kaggle. Otherwise leaves it None for A100.

### Unsloth $10K Special Track eligibility

If you want to compete in the Unsloth Special Track ($10,000 prize):
- Use Unsloth for fine-tuning (this notebook does)
- Document Unsloth-specific optimisations (4-bit QLoRA, gradient checkpointing="unsloth")
- Submit base vs fine-tuned A/B comparison (use unsloth-buddy demo HTML, or write your own)
